# Лекция 8. QR разложение и способы его вычисления. Линейная задача наименьших квадратов.

## В прошлый раз

- PLU разложение
- Ряд Неймана
- Число обусловленности

## План на сегодня

- Что такое QR разложение?
- Всегда ли оно существует?
- Как вычислить?

## Общая концепция матричных разложений

Вычислительная линейная алгебра занимается решением следующих задач:

- Решение линейных систем $Ax = f$
- Вычисление собственных значений и векторов
- Вычисление сингулярных значений и векторов 
- Вычисление обратных матриц и иногда детерминантов
- Вычисление **матричных функций** таких как $\exp(A), \cos(A)$ (это не поэлементные функции!)

Для решения таких задач мы представляемм матрицу в виде суммы и/или произведения матриц **более простой структуры**, таких что мы можем решить эти задачи быстрее и/или более устойчивым образом.

Что такое **более простая структура**?

## Матрицы простой структуры

Мы уже упоминали некоторые классы структурированных матриц. 

Для плотных матриц это следующие матрицы

- **унитарные матрицы**
- **нижне- верхнетреугольные матрицы** 
- **диагональные матрицы**

## QR разложение

- Как следует из названия, это представление матрицы в виде произведения

$$
    A = Q R, 
$$

где $Q$ – матрица с **ортогональными столбцами** и $R$ – **верхнетреугольная**.  

- Размеры матриц: $Q$ – $n \times m$, $R$ – $m \times m$, если $n\geq m$.

- QR разложение определено для любой **прямоугольной матрицы**.

## Приложения QR разложения

Это разложение играет ключевую роль при решении многих задач, например:
- Получение ортогонального базиса в линейном пространстве
- Используется для препроцессинга при вычислении SVD
- QR алгоритм для вычисления собственных векторов и собственных значений ([один из 10 самых важных алгоритмов ХХ века](https://archive.siam.org/pdf/news/637.pdf)) основан на вычислении QR разложения
- Решение переопределённых систем линейных уравнений

## Существование QR разложения

**Теорема.**

Каждая матрица $n \times m$ может быть представлена в виде QR разложения. 


Существует несколько способов доказательства и вычисления:

- Теоретический: используя матрицу Грама и разложение Холецкого
- Геометрический: используя ортогонализацию Грама-Шмидта
- Практический: используя преобразования Гивенса/Хаусхолдера

### Доказательство с использованием разложения Холецкого

Если у нас есть разложение

$$A = QR,$$

тогда $A^* A = ( Q R)^* (QR)  = R^* (Q^* Q) R = R^* R$, матрица $A^* A$ называется **матрицей Грама**, и её элементы – скалярные произведения столбцов матрицы $A$.  

### Случай матрицы полного ранга

Пусть $A$ имеет **полный столбцовый ранг**. Тогда легко показать, что $A^* A$ положительно определена:

$$
   (A^* A y, y) = (Ay, Ay) = \Vert Ay \Vert^2_2  > 0, \quad y\not = 0.
$$

Поэтому, $A^* A = R^* R$ всегда существует.

Тогда матрица $A R^{-1}$ унитарна:  

$$
   (A R^{-1})^* (AR^{-1})= R^{-*} A^* A R^{-1} = R^{-*} R^* R R^{-1} = I.
$$


### Случай матрицы неполного ранга

- QR разложение по-прежнему существует.

- Для любой матрицы неполного ранга существует последовательность матриц полного ранга $A_k$ такая что $A_k \rightarrow A$ (почему?).

- Каждая $A_k$ может быть разложена $A_k = Q_k R_k$. 

- Множество унитарных матриц образует компакт, таким образом найдётся сходящаяся подпоследовательность $Q_{n_k} \rightarrow Q$ (почему?), и $R_{n_k} = Q^*_{n_k} A_{n_k} \rightarrow Q^* A = R$, которая верхнетреугольная.

## Устойчивость вычисления QR разложения с помощью разложения Холецкого 

Итак, простейший способ вычисления QR разложения следующий

$$A^* A = R^* R,$$

и

$$Q = A R^{-1}.$$

Это **плохая идея** с точки зрения устойчивости. Покажем, почему это так.

In [1]:
import numpy as np
n = 50
r = 8
a = [[1.0 / (i + j + 0.5) for i in range(r)] for j in range(n)]
a = np.array(a)
print(a.shape)
q, Rmat = np.linalg.qr(a)
e = np.eye(r)
print('Built-in QR orth', np.linalg.norm(np.dot(q.T, q) - e))
gram_matrix = a.T.dot(a)
Rmat1 = np.linalg.cholesky(gram_matrix)
q1 = np.dot(a, np.linalg.inv(Rmat1.T))
print('Via Cholesky:', np.linalg.norm(np.dot(q1.T, q1) - e))

(50, 8)
Built-in QR orth 9.064926882123899e-16
Via Cholesky: 0.045879253193331665


## Второй способ: ортогонализация Грама-Шмидта

- QR разложение – это способ записи процесса ортогонализации Грама-Шмидта
- Для данного набора векторов $a_1, \ldots, a_m$ мы хотим найти ортонормированный базис $q_1, \ldots, q_m$ такой чтобы каждый вектор $a_i$ представлялся как линейная комбинация векторов из базиса.  

**Метод Грама-Шмидта:**
1. $q_1 := a_1/\Vert a_1 \Vert$
2. $q_2 := a_2 - (a_2, q_1) q_1, \quad q_2 := q_2/\Vert q_2 \Vert$
3. $q_3 := a_3 - (a_3, q_1) q_1 - (a_3, q_2) q_2, \quad q_3 := q_3/\Vert q_3 \Vert$
4. И так далее  

Заметим, что преобразование из $A$ в $Q$ имеет треугольную структуру, поскольку мы вычитаем из $k$-го вектора только предыдущие. Это следует из того, что произведение треугольных матриц – это треугольная матрица.

## Модифицированный метод Грама-Шмидта

- Метод Грама-Шмидта может быть очень неустойчивым (то есть генерировать векторы, которая не являются ортогональными, особенно если $q_k$ малой нормы).  
Это называется **потеря ортогональности**.  

- Этот недостаток метода Грама-Шмидта исправляется с помощью **модифицированного метода Грама-Шмидта**. Вместо вычисления

$$q_k := a_k - (a_k, q_1) q_1 - \ldots - (a_k, q_{k-1}) q_{k-1}$$

мы будем вычислять это выражение шаг за шагом. Сначала присвоим $q_k := a_k$, затем последовательно ортогонализуем:

$$
   q_k := q_k - (q_k, q_1)q_1, \quad q_k := q_{k} - (q_k,q_2)q_2, \ldots
$$

- В точной арифметике, это одинаковые алгоритмы. В неточной арифметике они абсолютно разные!

- Заметим, что сложность по-прежнему $\mathcal{O}(nm^2)$ операций

In [3]:
import numpy as np

n = 50
r = 8
a = [[1.0 / (i + j + 0.5) for i in range(r)] for j in range(n)]
A = np.array(a)

def gramm_schmidt(A):
    Q = np.zeros_like(A)
    R = np.zeros((A.shape[1], A.shape[1]))
    n = A.shape[1]
    for i in range(A.shape[1]):
        Q[:, i] = A[:, i].copy()
        for j in range(i):
            R[j, i] = Q[:, i] @ Q[:, j]
        for j in range(i):
            Q[:, i] -= R[j, i] * Q[:, j]
        R[i, i] = np.linalg.norm(Q[:, i])
        Q[:, i] /= np.linalg.norm(Q[:, i])
    return Q, R

Q, R = gramm_schmidt(A)
print(np.linalg.norm(Q.T @ Q - np.eye(r)))
print(np.linalg.norm(A - Q @ R))

def modified_gramm_schmidt(A):
    Q = np.zeros_like(A)
    R = np.zeros((A.shape[1], A.shape[1]))
    n = A.shape[1]
    for i in range(A.shape[1]):
        Q[:, i] = A[:, i].copy()
        for j in range(i):
            R[j, i] = Q[:, i] @ Q[:, j]
            Q[:, i] -= Q[:, i] @ Q[:, j] * Q[:, j]
        R[i, i] = np.linalg.norm(Q[:, i])
        Q[:, i] /= np.linalg.norm(Q[:, i])
    return Q, R

Q, R = modified_gramm_schmidt(A)
print(np.linalg.norm(Q.T @ Q - np.eye(r)))
print(np.linalg.norm(A - Q @ R))

0.1728832189878231
1.572773433245347e-16
1.2693399250207069e-09
1.6915110622252185e-16


## QR разложение: почти практический способ

Если $A = QR$, тогда  

$$
R = Q^* A,
$$

и нам нужно найти такую ортогональную матрицу $Q$, которая преобразует данную матрицу $A$ в верхнетреугольную.  
Для простоты мы будем смотреть на матрицы $n \times n$ такие что

$$ Q^* A = \begin{bmatrix} * & * & *  \\ 0 & * & * \\ 0 & 0 & * \\ & 0_{(n-m) \times m} \end{bmatrix} $$

Будем приводить матрицу к такому виду столбец за столбцом.

Сначала найдём такую матрицу Хаусхолдера $H_1 = (I - 2 uu^{\top})$ что 

$$ H_1 A = \begin{bmatrix} * & * & * \\ 0 & * & * \\ 0 & * & * \\ 0 & * & * \end{bmatrix} $$

Затем

$$ H_2 H_1 A = \begin{bmatrix} * & * & * \\ 0 & * & * \\ 0 & 0 & * \\ 0 & 0 & * \end{bmatrix}, $$

где

$$ H_2 = \begin{bmatrix} 1 & 0 \\ 0 & H'_2, \end{bmatrix} $$

и $H'_2$ матрица Хаусхолдера $3 \times 3$.

И наконец, 

$$ H_3 H_2 H_1 A = \begin{bmatrix} * & * & * \\ 0 & * & * \\ 0 & 0 & * \\ 0 & 0 & 0 \end{bmatrix}, $$

где $H_3=\begin{bmatrix}I_2 & \\ & {\widetilde H}_3 \end{bmatrix}$ такая что

$$ 
{\widetilde H}_3 \begin{bmatrix} \boldsymbol{\times} \\ \boldsymbol{\times}  \end{bmatrix} = 
\begin{bmatrix} \times \\ 0 \end{bmatrix}.
$$

Попробуйте самостоятельно реализовать такой алгоритм, это просто!

### Получение QR разложения

Так как 

$$ H_3H_2H_1A = HA = R,$$

где $H$ – унитарная матрица, то

$$ A = H^*R. $$

Таким образом $Q = H^*$.

## QR разложение: практический способ

- Поскольку мы работаем с плотной матрицей, то на практике нам нужен алгоритм, оперирующий блоками (почему?).  

- Вместо использования преобразования Хаусхолдера, мы будем использовать **блочное преобразование Хаусхолдера** вида 

$$H = (I - 2UU^*), $$

где $U^* U = I$.

## QR разложение: практический способ - 2

Аналогично преобразованию Хаусхолдера для вычисления QR разложения можно использовать преобразование Гивенса

$$\begin{bmatrix} \times & \times & \times \\ \bf{*} & \times & \times \\ \bf{*} & \times & \times \end{bmatrix} \to \begin{bmatrix} * & \times & \times \\ * & \times & \times \\ 0 & \times & \times \end{bmatrix} \to \begin{bmatrix} \times & \times & \times \\ 0 & * & \times \\ 0 & * & \times \end{bmatrix} \to \begin{bmatrix} \times & \times & \times \\ 0 & \times & \times \\ 0 & 0 & \times \end{bmatrix} $$

## Преобразование Гивенса vs. преобразование Хаусхолдера

- Матрицы Хаусхолдера полезны для плотных матриц (сложность примерно в два раза меньше), в которых необходимо занулить большое число элементов за одно отражение.
- Вращения Гивенса больше подходят для работы с разреженными матрицами или параллельными вычислениями, поскольку они зануляют только один элемент каждым действием.

## QR разложение: итоги

- Существует для любой матрицы
- Геометрически означает ортогонализацию векторов

# Линейная задача наименьших квадратов

## Переопределённые линейные системы

- Рассмотрим переопределённые линейные системы, в которых число уравнений больше, чем число неизвестных.
- Простейший пример: аппроксимация точек на плоскости с помощью линейной модели

Стандартный способ минимизации невязки (**линейная задача наименьших квадратов**)

$$\Vert A x - b \Vert_2 \rightarrow \min_x$$

## Переопределённая система и матрица Грама

Условие оптимальности $0\equiv \nabla \left(\|Ax-b\|_2^2\right)$, где $\nabla$ обозначает градиент. Поэтому,

$$
0 \equiv \nabla \left(\|Ax-b\|_2^2\right) = 2(A^*A x - A^*b) = 0.
$$

Таким образом,

$$
A^* A x = A^* b
$$

Матрица $A^* A$ называется **матрицей Грама**, а система называется **нормальным уравнением**. 

- Число обусловленности матрицы $A^* A$ равно квадрату числа обусловленности матрицы $A$ (проверьте!).
- Поэтому решать нормальное уравнение в таком виде – не самая хорошая идея!

## Псевдообратная матрица

Матрица $A^* A$ может быть вырождена в общем случае (почему?).
Поэтому необходимо ввести понятие псевдообратной матрицы  $A^{\dagger}$ такой что <br>
решение линейной задачи наименьших квадратов можно было записать в виде

$$x = A^{\dagger} b.$$

Матрица 

$$
A^{\dagger} = \lim_{\alpha \rightarrow 0}(\alpha I + A^* A)^{-1} A^*
$$ 

называется псевдообратной матрицей Мура-Пенроуза для матрицы $A$.

* Если матрица $A$ имеет полный ранг, тогда $A^* A$ невырождена, и мы получим $A^{\dagger} = \lim_{\alpha \rightarrow 0}(\alpha I + A^* A)^{-1} A^* = (A^* A)^{-1} A^*$.

* Если матрица $A$ квадратная и невырожденная, мы получим 

$$A^{\dagger} = \lim_{\alpha \rightarrow 0}(\alpha I + A^* A)^{-1} A^* = (A^* A)^{-1} A^* = A^{-1} A^{-*} A^* = A^{-1}$$ 

обычная обратная матрица для $A$

* Если $A$ имеет линейно зависимые столбцы, тогда $A^\dagger b$ даёт решение минимальной евклидовой нормы. 

## Вычисление псевдообратной матрицы с помощью SVD

Пусть $A = U \Sigma V^*$ SVD для матрицы $A$. Тогда,

$$A^{\dagger} = V \Sigma^{\dagger} U^*,$$

где $\Sigma^{\dagger}$ состоит из обращённых ненулевых сингулярных чисел матрицы $A$. Действительно,

$$A^{\dagger} = \lim_{\alpha \rightarrow 0}(\alpha I + A^* A)^{-1} A^* = \lim_{\alpha \rightarrow 0}( \alpha VV^* + V \Sigma^2 V^*)^{-1} V \Sigma U^* = \lim_{\alpha \rightarrow 0}( V(\alpha I + \Sigma^2) V^*)^{-1} V \Sigma U^* = V \lim_{\alpha \rightarrow 0}(\alpha I + \Sigma^2)^{-1} \Sigma U^* = V \Sigma^{\dagger} U^*,$$

* Вы можете проверить, что $\Sigma^{\dagger}$ состоит из обращённых ненулевых сингулярных чисел <br>
* Если сингулярные числа малы, их можно не обращать. Это даст решение менее чувствительное к шуму в правой части

**Q:** что произошло с числом обусловленности?

## Стандартный способ решения линейной задачи наименьших квадратов

Использование $QR$ разложения.

Любая матрица может быть представлена в виде 

$$
A = Q R,
$$

где $Q$ – унитарная матрица, и $R$ – верхнетреугольная.

Тогда, если $A$ имеет полный ранг, тогда

$$
x = A^{\dagger}b = (A^*A)^{-1}A^*b = ((QR)^*(QR))^{-1}(QR)^*b = (R^*Q^*QR)^{-1}R^*Q^*b = R^{-1}Q^*b. 
$$ 

Таким образом, задача поиска оптимального $x$ эквивалентна решению следующей квадратной системы 

$$
Rx = Q^* b.
$$

Так как $R$ верхнетреугольная, решение этой системы требует $\mathcal{O}(n^2)$ операций. Также этот способ более устойчив, чем использование псевдообратной матрицы напрямую.

# Выводы про задачу наименьших квадратов

- Нормальное уравнение
- Решение с помощью QR разложения
- Псевдообратная матрица: определение и способ вычисления